# Valor del cliente y segmentación por riesgo de abandono

Dataset: `Customer-Churn-Records-clean.csv`.

Método: matriz **valor × riesgo** y **pérdida esperada** (`P(abandono) × Balance`). El modelo de riesgo **no usa** `Complain` (fuga con `Exited`). Ejecutar las celdas en orden.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("Customer-Churn-Records-clean.csv")
print(df.shape)
print(df.columns.tolist())

In [ ]:
# --- Valor: depósitos (Balance) y ranking de cartera ---
df["ValueRank"] = df["Balance"].rank(method="average", pct=True)
df["ValueTier"] = pd.qcut(
    df["Balance"].rank(method="first"),
    3,
    labels=["Valor bajo", "Valor medio", "Valor alto"],
)

print("=== VALOR (Balance) ===")
print(df.groupby("ValueTier", observed=True)["Balance"].describe())
print("\nSaldo total por ValueTier:")
print(df.groupby("ValueTier", observed=True)["Balance"].sum())
print("\nChurn observado por ValueTier:")
print(
    df.groupby("ValueTier", observed=True).agg(
        n=("Exited", "size"),
        churners=("Exited", "sum"),
        churn_rate=("Exited", "mean"),
        balance_sum=("Balance", "sum"),
        balance_mean=("Balance", "mean"),
    )
)

In [ ]:
# --- Riesgo: P(Exited) con logística; sin Complain ni IDs ---
id_cols = [c for c in ["RowNumber", "CustomerId", "Surname"] if c in df.columns]
leak_cols = [c for c in ["Complain"] if c in df.columns]
derived = [c for c in ["AgeGroup", "BalanceCategory", "ValueRank", "ValueTier"] if c in df.columns]

y = df["Exited"].astype(int)
X = df.drop(columns=id_cols + leak_cols + derived + ["Exited"])

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
if "NumOfProducts" in X.columns:
    cat_cols = sorted(set(cat_cols + ["NumOfProducts"]))
num_cols = [c for c in X.columns if c not in cat_cols]

X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=True)
feature_names = X_enc.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y, test_size=0.30, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
X_all_s = scaler.transform(X_enc)

model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
model.fit(X_train_s, y_train)

p_train = model.predict_proba(X_train_s)[:, 1]
p_test = model.predict_proba(X_test_s)[:, 1]
df["p_churn"] = model.predict_proba(X_all_s)[:, 1]

print("=== RENDIMIENTO DEL MODELO ===")
print("AUC train:", roc_auc_score(y_train, p_train))
print("AUC test:", roc_auc_score(y_test, p_test))
print("Brier test:", brier_score_loss(y_test, p_test))
print("\nReporte test (umbral 0.50):")
print(classification_report(y_test, (p_test >= 0.50).astype(int), digits=3))

coef = pd.Series(model.coef_[0], index=feature_names).sort_values(key=np.abs, ascending=False)
print("\nCoeficientes (estandarizados, top 15 por |beta|):")
print(coef.head(15).to_string())

print("\n=== CALIBRACIÓN POR DECILES (test) ===")
dec = pd.qcut(p_test, 10, labels=False, duplicates="drop")
calib = (
    pd.DataFrame({"p": p_test, "y": y_test.to_numpy(), "decile": dec})
    .groupby("decile")
    .agg(n=("y", "size"), p_mean=("p", "mean"), churn_obs=("y", "mean"))
)
print(calib.to_string())

In [ ]:
# --- Segmentación valor × riesgo y pérdida esperada ---
df["RiskTier"] = pd.qcut(
    df["p_churn"].rank(method="first"),
    3,
    labels=["Riesgo bajo", "Riesgo medio", "Riesgo alto"],
)
df["expected_loss"] = df["p_churn"] * df["Balance"]

def _label(row):
    v, r = row["ValueTier"], row["RiskTier"]
    if v == "Valor alto" and r == "Riesgo alto":
        return "Proteger (alto valor, alto riesgo)"
    if v == "Valor alto" and r == "Riesgo medio":
        return "Vigilar valor (alto valor, riesgo medio)"
    if v == "Valor alto" and r == "Riesgo bajo":
        return "Cultivar (alto valor, bajo riesgo)"
    if v == "Valor medio" and r == "Riesgo alto":
        return "Retener selectivo (valor medio, alto riesgo)"
    if v == "Valor bajo" and r == "Riesgo alto":
        return "Gestionar coste (bajo valor, alto riesgo)"
    if r == "Riesgo medio":
        return "Estándar (riesgo medio)"
    return "Low-touch (bajo riesgo)"

df["Segment"] = df.apply(_label, axis=1)

print("=== MATRIZ VALOR × RIESGO (n, churn, saldo, pérdida esperada) ===")
matrix = (
    df.groupby(["ValueTier", "RiskTier"], observed=True)
    .agg(
        n=("Exited", "size"),
        churners=("Exited", "sum"),
        churn_rate=("Exited", "mean"),
        p_churn_mean=("p_churn", "mean"),
        balance_sum=("Balance", "sum"),
        expected_loss=("expected_loss", "sum"),
    )
    .reset_index()
)
matrix["share_customers"] = matrix["n"] / matrix["n"].sum()
matrix["share_churners"] = matrix["churners"] / matrix["churners"].sum()
matrix["share_balance"] = matrix["balance_sum"] / matrix["balance_sum"].sum()
matrix["share_expected_loss"] = matrix["expected_loss"] / matrix["expected_loss"].sum()
print(matrix.to_string(index=False))

print("\n=== CHURN OBSERVADO: filas=valor, columnas=riesgo ===")
print(
    df.pivot_table(
        index="ValueTier", columns="RiskTier", values="Exited", aggfunc="mean", observed=True
    )
)
print("\n=== n: filas=valor, columnas=riesgo ===")
print(
    df.pivot_table(
        index="ValueTier", columns="RiskTier", values="Exited", aggfunc="size", observed=True
    )
)
print("\n=== PÉRDIDA ESPERADA (suma de p_churn × Balance) ===")
print(
    df.pivot_table(
        index="ValueTier",
        columns="RiskTier",
        values="expected_loss",
        aggfunc="sum",
        observed=True,
    )
)

In [ ]:
# --- Perfil de cada segmento de acción ---
print("=== SEGMENTOS DE ACCIÓN ===")
seg = (
    df.groupby("Segment", observed=True)
    .agg(
        n=("Exited", "size"),
        churners=("Exited", "sum"),
        churn_rate=("Exited", "mean"),
        p_churn_mean=("p_churn", "mean"),
        age_mean=("Age", "mean"),
        products_mean=("NumOfProducts", "mean"),
        active_rate=("IsActiveMember", "mean"),
        balance_sum=("Balance", "sum"),
        expected_loss=("expected_loss", "sum"),
    )
    .sort_values("expected_loss", ascending=False)
)
seg["share_expected_loss"] = seg["expected_loss"] / seg["expected_loss"].sum()
seg["share_churners"] = seg["churners"] / seg["churners"].sum()
print(seg.to_string())

print("\n=== MIX GEOGRÁFICO (% filas del segmento) ===")
print(pd.crosstab(df["Segment"], df["Geography"], normalize="index").round(3))

print("\n=== MIX GÉNERO ===")
print(pd.crosstab(df["Segment"], df["Gender"], normalize="index").round(3))

print("\n=== MIX NÚMERO DE PRODUCTOS ===")
print(pd.crosstab(df["Segment"], df["NumOfProducts"], normalize="index").round(3))

print("\n=== TOP 15 CLIENTES POR PÉRDIDA ESPERADA ===")
top = df.nlargest(15, "expected_loss")[
    [
        "CustomerId",
        "Geography",
        "Gender",
        "Age",
        "Balance",
        "NumOfProducts",
        "IsActiveMember",
        "p_churn",
        "expected_loss",
        "Segment",
        "Exited",
    ]
]
print(top.to_string(index=False))

out_cols = [
    "CustomerId",
    "Geography",
    "Gender",
    "Age",
    "Balance",
    "NumOfProducts",
    "IsActiveMember",
    "ValueTier",
    "p_churn",
    "RiskTier",
    "expected_loss",
    "Segment",
    "Exited",
]
df[out_cols].to_csv("Customer-Risk-Value-Segments.csv", index=False)
print("\nGuardado: Customer-Risk-Value-Segments.csv")